# Semi-supervised learning
**Machine Learning · Week 6 (Learning without labels) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/ml/w6d5.html

Only **4 of 300** points have labels. A normal classifier trained on those 4 does poorly.
Semi-supervised methods also use the 296 unlabeled points, whose *shape* reveals where the classes are.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=300, noise=0.08, random_state=0)
rng = np.random.default_rng(4)
labeled = np.concatenate([rng.choice(np.where(y == c)[0], 2, replace=False) for c in [0, 1]])

y_partial = np.full_like(y, -1)          # scikit-learn's convention: -1 means "no label"
y_partial[labeled] = y[labeled]
print("labeled points:", len(labeled), "of", len(X))

## Supervised learning with only 4 labels

In [ ]:
from sklearn.svm import SVC

supervised = SVC(kernel="rbf", gamma=1.0).fit(X[labeled], y[labeled])
print("accuracy on all 300 points:", round(supervised.score(X, y), 3))

## Label spreading
Connect every point to its 7 nearest neighbors, then let the 4 known labels flow along the connections,
like ink spreading through wet paper. Labels travel easily *within* a moon and hardly at all across the gap.

In [ ]:
from sklearn.semi_supervised import LabelSpreading

# alpha = how much each point listens to its neighbors (0.9 lets labels travel far along the moon)
spreading = LabelSpreading(kernel="knn", n_neighbors=7, alpha=0.9, max_iter=1000).fit(X, y_partial)
guessed = spreading.transduction_                # the label it settled on for every point
print("accuracy on all 300 points:", round((guessed == y).mean(), 3))

In [ ]:
colors = np.array(["#ff8a50", "#5577f5"])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True, layout="constrained")
xx, yy = np.meshgrid(np.linspace(-1.4, 2.4, 200), np.linspace(-0.9, 1.4, 150))
axes[0].contourf(xx, yy, supervised.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape), levels=[-0.5, 0.5, 1.5], colors=["#ffe3d6", "#dfe6ff"])
axes[0].scatter(X[:, 0], X[:, 1], c="#cfcadb", s=10)
axes[0].set_title("Supervised, 4 labels only")
axes[1].scatter(X[:, 0], X[:, 1], c=colors[guessed], s=10)
axes[1].set_title("Label spreading, 4 labels + 296 unlabeled")
for ax in axes:
    ax.scatter(X[labeled, 0], X[labeled, 1], c=colors[y[labeled]], marker="*", s=380, edgecolors="#2c2842", linewidths=1.2, zorder=3)
plt.show()

## Self-training, by hand (and its trap)
Another classic: train on what you have, label the unlabeled points the model is *most* sure about,
add them to the training set, and repeat. Watch the accuracy carefully.

In [ ]:
X_lab, y_lab = X[labeled], y[labeled]
pool = np.setdiff1d(np.arange(len(X)), labeled)
for round_number in range(6):
    model = SVC(kernel="rbf", gamma=1.0).fit(X_lab, y_lab)
    print(f"round {round_number}: trained on {len(y_lab):3d} labels -> accuracy on all points {model.score(X, y):.3f}")
    scores = model.decision_function(X[pool])
    confident = pool[np.argsort(-np.abs(scores))[:40]]          # the 40 most confident guesses
    X_lab = np.vstack([X_lab, X[confident]])
    y_lab = np.concatenate([y_lab, model.predict(X[confident])])   # pseudo-labels, not true labels!
    pool = np.setdiff1d(pool, confident)

It never improves, and slowly gets worse. The model's early boundary is wrong near the tips of the moons,
and every round it labels more points according to that wrong boundary, then trains on its own mistakes.
This is **confirmation bias**, the classic danger of self-training. Label spreading did much better here
because it follows the shape of the data, not the first model's guesses.

## Your turn: how many labels do you need?
For 1, 2, 5, 10 and 25 labels per class, compare supervised-only (an SVM on the labeled points) with
label spreading. Labels are drawn at random, so average over 10 random draws.

In [ ]:
from sklearn.semi_supervised import LabelSpreading
from sklearn.svm import SVC

def accuracies(n_per_class, seed):
    rng = np.random.default_rng(seed)
    lab = np.concatenate([rng.choice(np.where(y == c)[0], n_per_class, replace=False) for c in [0, 1]])
    partial = np.full_like(y, -1)
    partial[lab] = y[lab]
    supervised_acc = None   # TODO: SVC(kernel="rbf", gamma=1.0) fitted on X[lab], y[lab]; its .score(X, y)
    spreading_acc = None    # TODO: LabelSpreading(kernel="knn", n_neighbors=7, alpha=0.9, max_iter=1000).fit(X, partial),
                            #       then the share of transduction_ that equals y
    return supervised_acc, spreading_acc

for n in [1, 2, 5, 10, 25]:
    print(n, [accuracies(n, seed) for seed in range(3)])

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
from sklearn.semi_supervised import LabelSpreading
from sklearn.svm import SVC

def accuracies(n_per_class, seed):
    rng = np.random.default_rng(seed)
    lab = np.concatenate([rng.choice(np.where(y == c)[0], n_per_class, replace=False) for c in [0, 1]])
    partial = np.full_like(y, -1)
    partial[lab] = y[lab]
    supervised_acc = SVC(kernel="rbf", gamma=1.0).fit(X[lab], y[lab]).score(X, y)
    spreading = LabelSpreading(kernel="knn", n_neighbors=7, alpha=0.9, max_iter=1000).fit(X, partial)
    spreading_acc = (spreading.transduction_ == y).mean()
    return supervised_acc, spreading_acc

budgets = [1, 2, 5, 10, 25]
means = np.array([np.mean([accuracies(n, seed) for seed in range(10)], axis=0) for n in budgets])
for n, (sup, spr) in zip(budgets, means):
    print(f"{n:2d} labels per class: supervised only {sup:.3f}, label spreading {spr:.3f}")

plt.plot(budgets, means[:, 0], "o-", color="#ff8a50", label="supervised only")
plt.plot(budgets, means[:, 1], "o-", color="#17a98c", label="label spreading")
plt.xscale("log")
plt.xticks(budgets, [str(b) for b in budgets])
plt.xlabel("labels per class")
plt.ylabel("accuracy on all 300 points")
plt.legend()
plt.show()
print("With a few labels, the unlabeled points' shape is worth a lot. With many labels, the gap closes.")